<a href="https://colab.research.google.com/github/alscop/ESAA-26-1/blob/main/ESAA_OB_week03_1_Review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 수상작 리뷰

# 수상작 소개

**주차수요 예측 AI 경진대회**  

(https://drive.google.com/open?id=1auJixB7o4cn8RsQTWNcEFkqoYyccizhz&usp=drive_copy)

목표: 유형별 임대주택 설계 시 단지 내 적정 주차 수요를 예측

설명; 아파트 단지 내 필요한 주차대수는 ①법정주차대수 ②장래주차수요 중 큰 값에 따라 결정하게되어 있어, 정확한 ②장래주차수요의 산정을 필요로 함

현재 ②장래주차수요는 ‘주차원단위’와 ‘건축연면적’을 기초로하여 산출되고 있으며, ‘주차원단위’는 신규 건축예정 부지 인근의 유사 단지를 피크 시간대 방문하여 주차된 차량대수를 세는 방법으로 조사하고 있음

이 경우 인력조사로 인한 오차발생, 현장조사 시점과 실제 건축시점과의 시간차 등의 문제로 과대 또는 과소 산정의 가능성을 배제할 수 없음

평가: MAE(Mean Absolute Error)



---


## 코드 흐름 요약

- 데이터 중복 처리
- 변수화, 매핑
- Catboost 단일 모델 사용

### 코드 흐름

1. 라이브러리 임포트
- 데이터 핸들링 툴
- 데이터 시각화 툴
- 한글 폰트 설정
- 그래프 마이너스 표시 설정


2. 데이터 로드
- info 확인
- NA 확인

*EDA 시 이상치 있으나 데이터 샘플 수 적어 이상치 제거 안 함

3. 데이터 전처리
- 임대보증금/임대료 object에서 float로 타입 변경
-  NULL 처리
  - 임대보증금/임대료: 공급유형과 자격유형 value_counts()로 분포 확인 -> 0 대체
  - 지하철역수/버정 수: 0 대체
  - 자격유형: 동일 단지코드 내 자격유형 분포 확인 후 겹치는 값으로 대체
- 중복 확인
  - `duplicates().shape`로 중복 데이터 규모 확인, 제거
  - 단지코드 `nunique()`로 단지수 확인
  - 단지코드로 `groupby([''])` 후 각 칼럼 고유값 확인 -> 단지수보다 큰 경우 두 개 이상의 항목 존재하는 것.
- 단지코드별 집계
  - 1단지코드 1값 변수
  - 1단지코드 값 2이상 변수: 도메인 지식 활용해 처리



4. 모델링
- Catboost
- 평가 지수: MAE

## 새롭게 알게 된 내용 / 어려운 내용 / 배울 점 등 정리 및 주요 코드 3줄 이상 작성

- 데이터 오류 존재하는 데이콘이라 전처리 과정 더 복잡해보임.
- 하나의 단지코드에 대해 둘 이상의 항목이 존재할 경우 변수로 만들어 사용


In [1]:
# print(f"단지코드 C2483에서 유일한 값을 가지는 변수들:\n{list(train.columns[train[train.단지코드=='C2483'].nunique()==1])}")

`단지코드 C2483에서 유일한 값을 가지는 변수들:`  
`['단지코드', '총세대수', '임대건물구분', '지역', '공급유형', '공가수', '자격유형', '도보 10분거리 내 지하철역 수(환승노선 수 반영)', '도보 10분거리 내 버스정류장 수', '단지내주차면수', '등록차량수']`

In [2]:
# train.groupby(['단지코드']).nunique(dropna=False)

`423 rows × 14 columns`

In [3]:
# train.groupby(['단지코드']).nunique(dropna=False).sum(axis=0)

`총세대수                             423`  
`임대건물구분                           456`  
`지역                               423`  
`공급유형                             488`  
`전용면적                            1898`  
`전용면적별세대수                        2230`  
`공가수                              423`  
`자격유형                             510`  
`임대보증금                           1277`  
`임대료                             1289`  
`도보 10분거리 내 지하철역 수(환승노선 수 반영)     423`  
`도보 10분거리 내 버스정류장 수               423`  
`단지내주차면수                          423`  
`등록차량수                            423`  
`dtype: int64`

- 값이 423보다 크면 하나의 단지코드에 대해 둘 이상의 항목이 존재하는 것을 의미.  
- 단지코드별 집계시 총세대수, 지역, 공가수, 도보 10분거리 내 지하철역 수(환승노선 수 반영), 도보 10분거리 내 버스정류장 수, 단지내주차면수, 등록차량수는 그대로 사용(423이라 하나의 단지코드에 하나의 항목 존재)
- 임대건물구분, 공급유형, 전용면적, 전용면적별세대수, 자격유형, 임대보증금, 임대료는 하나의 단지코드에 대해 둘 이상의 항목 존재
- 이 변수들은 **각 항목들을 변수로 만들어 사용**하는 것이 좋아보임

---
변수들 항목 살펴보고 도메인지식/외부 정보 찾아 매핑해준 과정이 인상깊음 ...

---
앙상블 없이 Catboost 단일모델 사용했음에도 성능 좋았음. 전처리 과정이 큰 도움이 된 것으로 보임  
**범주형 변수 인코딩을 해주는 내부 로직**이 있어서 **Catboost** 사용 많이 하는 듯